# AxiraNews — ML Recommendation Simulation
Simulates user vector drift, feed ranking scores, and serendipity injection over time.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import softmax

plt.style.use('dark_background')
plt.rcParams.update({'axes.facecolor': '#0A0A0C', 'figure.facecolor': '#070708',
                     'text.color': '#F0EDE6', 'axes.labelcolor': '#C8A97E',
                     'xtick.color': '#4A4A56', 'ytick.color': '#4A4A56',
                     'axes.edgecolor': '#252528', 'grid.color': '#1A1A1E'})

CATEGORIES    = ['cybersecurity', 'finance', 'global', 'science', 'local', 'tech']
SOURCE_TYPES  = ['gov', 'security', 'mainstream', 'localMedia', 'companyPR']
N_ARTICLES    = 200
N_DAYS        = 30
EPSILON       = 0.12  # serendipity injection rate

# Event weights (matches production ranking.ts)
WEIGHTS = {'like': 4, 'save': 3, 'dwell': 2, 'click': 1, 'impression': 0.1}
print('Config loaded ✓')

In [ ]:
# ── Simulate a user reading session over N_DAYS ──────────────────────────────
rng = np.random.default_rng(42)

# True user preference (hidden ground truth)
true_pref = np.array([0.40, 0.20, 0.15, 0.10, 0.10, 0.05])  # loves cyber & finance

def simulate_user_vector(true_pref, n_days=N_DAYS, events_per_day=15):
    """Simulate how the recommendation engine learns user preferences."""
    vectors, entropies, serendipity_hits = [], [], []
    accumulated = np.zeros(len(CATEGORIES))

    for day in range(n_days):
        # User reads articles — noisy signal around true preference
        for _ in range(events_per_day):
            cat_idx = rng.choice(len(CATEGORIES), p=true_pref)
            event   = rng.choice(list(WEIGHTS.keys()), p=[0.05, 0.08, 0.30, 0.22, 0.35])
            accumulated[cat_idx] += WEIGHTS[event]

        # Normalize to distribution
        vec = accumulated / (accumulated.sum() + 1e-9)
        vectors.append(vec.copy())

        # Shannon entropy — measures how diverse the user's taste is
        e = -np.sum(vec * np.log(vec + 1e-9))
        entropies.append(e)

        # Serendipity: inject random cat when entropy is low (user is stuck)
        hit = 1 if (e < 1.2 and rng.random() < EPSILON) else 0
        serendipity_hits.append(hit)

    return np.array(vectors), np.array(entropies), np.array(serendipity_hits)

vectors, entropies, seren = simulate_user_vector(true_pref)
print(f'Simulated {N_DAYS} days | Final vector: {dict(zip(CATEGORIES, vectors[-1].round(3)))}')

In [ ]:
# ── Plot 1: User vector drift over time ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('AxiraNews — Recommendation Engine Simulation', color='#C8A97E', fontsize=14, fontweight='bold')

COLORS = ['#FF5F57', '#28C840', '#5AC8FA', '#B48EAD', '#FEBC2E', '#C8A97E']

# User vector drift
ax = axes[0, 0]
for i, cat in enumerate(CATEGORIES):
    ax.plot(vectors[:, i], label=cat, color=COLORS[i], linewidth=1.8, alpha=0.9)
ax.axhline(y=true_pref[0], color='#FF5F57', linestyle='--', alpha=0.3, linewidth=1)
ax.set_title('User Vector Convergence', color='#F0EDE6')
ax.set_xlabel('Day'); ax.set_ylabel('Weight')
ax.legend(fontsize=7, framealpha=0.3)
ax.grid(True, alpha=0.2)

# Entropy over time
ax = axes[0, 1]
ax.fill_between(range(N_DAYS), entropies, alpha=0.3, color='#C8A97E')
ax.plot(entropies, color='#C8A97E', linewidth=2)
ax.axhline(y=1.2, color='#FEBC2E', linestyle='--', alpha=0.6, label='serendipity threshold')
for d in np.where(seren)[0]:
    ax.axvline(x=d, color='#28C840', alpha=0.4, linewidth=1)
ax.set_title('Preference Entropy + Serendipity Triggers', color='#F0EDE6')
ax.set_xlabel('Day'); ax.set_ylabel('Shannon Entropy')
ax.legend(fontsize=8, framealpha=0.3)
ax.grid(True, alpha=0.2)

# Final vector vs ground truth
ax = axes[1, 0]
x = np.arange(len(CATEGORIES))
bars = ax.bar(x - 0.2, true_pref, 0.35, label='True preference', color='#4A4A56', alpha=0.8)
bars2 = ax.bar(x + 0.2, vectors[-1], 0.35, label='Learned vector', color='#C8A97E', alpha=0.9)
ax.set_title('Learned vs True Preference (Day 30)', color='#F0EDE6')
ax.set_xticks(x); ax.set_xticklabels(CATEGORIES, rotation=30, ha='right', fontsize=8)
ax.legend(fontsize=8, framealpha=0.3); ax.grid(True, alpha=0.2, axis='y')
l1 = np.linalg.norm(vectors[-1] - true_pref, 1)
ax.set_xlabel(f'L1 distance from truth: {l1:.3f}')

# Score distribution for a ranked feed
ax = axes[1, 1]
n = 50
freshness  = np.exp(-rng.uniform(0, 48, n) / 12)           # exponential decay
relevance  = vectors[-1][rng.choice(len(CATEGORIES), n)]   # user vector match
credibility = rng.beta(5, 2, n)                             # source quality
scores = 0.40 * relevance + 0.35 * freshness + 0.25 * credibility
ax.scatter(freshness, relevance, c=scores, cmap='YlOrBr', s=40, alpha=0.85, edgecolors='none')
ax.set_title('Feed Ranking Score Distribution (50 articles)', color='#F0EDE6')
ax.set_xlabel('Freshness'); ax.set_ylabel('Relevance')
ax.grid(True, alpha=0.2)
sm = plt.cm.ScalarMappable(cmap='YlOrBr'); sm.set_array(scores)
plt.colorbar(sm, ax=ax, label='Final Score')

plt.tight_layout()
plt.savefig('recommendation_sim.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Serendipity injected {seren.sum()} times over {N_DAYS} days')